# Assignment 4: Building, and critically interpreting concept bottleneck models for skin lesion diagnosis prediction

**Responsible AI 2026**

In this project, you will build an intrinsically explainable concept bottleneck model (CBM) for diagnosing skin lesions (from photography) into five different cancer-related diagnostic categories. To this end, you will utilize the derm7pt dataset, which is annotated with seven check-list criteria (the 7-pt criteria) that you will use to form intermediate concepts for your model.

Upon building the CBM, you will establish a validation pipeline to assess the quality not just of the model, but also of its intrinsic explanations. Moreover, you will add further layers of interpretation and inspection to the model and extend your validation pipeline to also critically assess these.

**Note:** Parts 2-3 will be released in the upcoming weeks. Again, your project should be presented in the form of a poster.

## Part 1: Building the CBM and creating explanation success criteria

- Download the derm7pt dataset and use an AI tool of your choice to build a concept bottleneck model that utilizes the 7-pt criteria as an intermediate concept layer and predicts the diagnostic labels. Make sure to verify that you can read the resulting code and that you find the resulting model to be in line with the definition of a concept bottleneck; you will be responsible for its correctness.
- Set your model up so that it can be trained either with independent, sequential or joint training. Again, make sure that this is done correctly, and ensure that you can account for the data allocated to training, validation and test, and that you agree with the choices.
- Train the model using all three regimes, and perform a standard assessment of the models’ performances. What do you see?
- Perform an exploratory study of the explanations given by the concepts: Do they support any performance differences that you observe between the three training regimes? What happens if you intervene on the concepts as in the CBM paper?
- In discussion with your group members, suggest 3 potential use cases of model explanations for this type of prediction model. Summarize your discussion into a list of use cases and corresponding measurable success criteria.
    - Example: Usecase - Get patients to seek healthcare when needed. Success criterion - Patients using the AI tool are better at seeking healthcare when they need it, and staying away when they don’t.

# Part 1: Solution

**How this notebook is laid out**

1. Setup and data
2. The model, and checks that it really is a concept bottleneck
3. Training: independent, sequential and joint
4. Standard performance assessment
5. Exploring the explanations
6. Interventions
7. Use cases and success criteria

**Before you run it:** the dataset needs a password.

1. Fill in the [request form](https://forms.gle/iPnCEVkUKYwWYALh9) linked from the [download page](https://derm.cs.sfu.ca/Download.html). The username and password are shown when you finish.
2. Download `release_v0.zip` and unzip it so that this path exists: `project4/data/release_v0/meta/meta.csv`.
3. Use the `.venv` kernel for this notebook and choose **Run All**.

The data licence is CC BY-NC-ND: non-commercial use, and no sharing of changed versions. The dataset's README also says the images may not be redistributed. Section 5.5 shows lesion images, so check this before sharing the notebook once it has been run.

## 1. Setup and data

All the settings are in the next cell. Trained models are saved in `checkpoints/`, so running the notebook again does not retrain them. **If you change a training setting, delete that folder.**

`IMAGE_TYPE` picks the image: `"derm"` is a dermoscopy photo (taken through a skin magnifier) and `"clinic"` is an ordinary close-up photo. The default is `"derm"` because the 7-point criteria are defined on dermoscopy images. In 9 cases the clinical photo is missing and the dataset uses the dermoscopy image instead, which matters only if you choose `"clinic"`.

In [ ]:
import copy
import hashlib
import random
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.transforms.functional as TF
from IPython.display import display
from PIL import Image
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, balanced_accuracy_score, confusion_matrix, f1_score, roc_auc_score
from torch.utils.data import DataLoader, Dataset
from torchvision import models
from torchvision.transforms import v2

DATA_DIR = Path("data/release_v0")
IMAGE_TYPE = "derm"            # "derm" or "clinic"
SEEDS = [0, 1, 2]              # each model is trained once per seed
EPOCHS = 30
BATCH_SIZE = 32
LR_BACKBONE = 1e-4             # the pretrained part of the network
LR_NEW_LAYERS = 1e-3           # the new last layer of the ResNet
LR_F = 1e-2                    # f in the joint model: its inputs are small (0 to 1), so it needs bigger steps
WEIGHT_DECAY = 1e-4
LAMBDAS = [0.1, 1.0, 10.0]     # joint model: how much the concept loss counts
N_RANDOM_ORDERS = 20           # interventions: how many random orders to average

CACHE_DIR = Path("checkpoints") / IMAGE_TYPE
CACHE_DIR.mkdir(parents=True, exist_ok=True)
DEVICE = "mps" if torch.backends.mps.is_available() else "cuda" if torch.cuda.is_available() else "cpu"
print("Running on:", DEVICE)

# Fixed colors for the charts, so each model looks the same in every figure
COLORS = {"independent": "#2a78d6", "sequential": "#eb6834", "joint": "#1baf7a", "standard": "#52514e"}
plt.rcParams.update({"axes.spines.top": False, "axes.spines.right": False, "axes.grid": True,
                     "grid.color": "#e8e7e3", "axes.axisbelow": True, "figure.dpi": 110})


def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

### The labels

The raw data has more detailed labels than we need. We use the grouping from the dataset authors ([Kawahara et al.](https://github.com/jeremykawahara/derm7pt)), which gives **5 diagnoses** and **7 criteria with 19 possible values** in total. Each criterion is one concept group, and each lesion has exactly one value per group.

`CHECKLIST_POINTS` holds the points from the original 7-point checklist. We use them later to compare the model with the rule doctors use.

In [ ]:
DIAGNOSES = {
    "BCC": ["basal cell carcinoma"],
    "NEV": ["nevus", "blue nevus", "clark nevus", "combined nevus", "congenital nevus", "dermal nevus",
            "recurrent nevus", "reed or spitz nevus"],
    "MEL": ["melanoma", "melanoma (in situ)", "melanoma (less than 0.76 mm)", "melanoma (0.76 to 1.5 mm)",
            "melanoma (more than 1.5 mm)", "melanoma metastasis"],
    "MISC": ["DF/LT/MLS/MISC", "dermatofibroma", "lentigo", "melanosis", "miscellaneous", "vascular lesion"],
    "SK": ["seborrheic keratosis"],
}
CONCEPTS = {
    "pigment_network": {"absent": ["absent"], "typical": ["typical"], "atypical": ["atypical"]},
    "blue_whitish_veil": {"absent": ["absent"], "present": ["present"]},
    "vascular_structures": {"absent": ["absent"],
                            "regular": ["regular", "arborizing", "comma", "hairpin", "within regression", "wreath"],
                            "irregular": ["dotted/irregular", "dotted", "linear irregular"]},
    "pigmentation": {"absent": ["absent"],
                     "regular": ["regular", "diffuse regular", "localized regular"],
                     "irregular": ["irregular", "diffuse irregular", "localized irregular"]},
    "streaks": {"absent": ["absent"], "regular": ["regular"], "irregular": ["irregular"]},
    "dots_and_globules": {"absent": ["absent"], "regular": ["regular"], "irregular": ["irregular"]},
    "regression_structures": {"absent": ["absent"], "present": ["present", "blue areas", "white areas", "combinations"]},
}
CHECKLIST_POINTS = {("pigment_network", "atypical"): 2, ("blue_whitish_veil", "present"): 2,
                    ("vascular_structures", "irregular"): 2, ("pigmentation", "irregular"): 1,
                    ("streaks", "irregular"): 1, ("dots_and_globules", "irregular"): 1,
                    ("regression_structures", "present"): 1}

CLASS_NAMES = list(DIAGNOSES)
GROUP_NAMES = list(CONCEPTS)
N_CLASSES, N_GROUPS = len(CLASS_NAMES), len(GROUP_NAMES)
MEL = CLASS_NAMES.index("MEL")

# SLICES says which positions in the 19-long concept vector belong to which criterion
SLICES, VALUE_NAMES, start = [], [], 0
for group, values in CONCEPTS.items():
    SLICES.append(slice(start, start + len(values)))
    VALUE_NAMES += [f"{group}: {value}" for value in values]
    start += len(values)
N_VALUES = start
POINTS = np.array([CHECKLIST_POINTS.get(tuple(name.split(": ")), 0) for name in VALUE_NAMES])
print(f"{N_CLASSES} diagnoses, {N_GROUPS} criteria, {N_VALUES} concept values")

In [ ]:
assert (DATA_DIR / "meta/meta.csv").exists(), (
    f"Dataset not found in {DATA_DIR.resolve()}. See the download steps at the top of Part 1.")
meta = pd.read_csv(DATA_DIR / "meta/meta.csv")

diagnosis_to_number = {name: i for i, names in enumerate(DIAGNOSES.values()) for name in names}
Y_TRUE = meta["diagnosis"].map(diagnosis_to_number)
assert Y_TRUE.notna().all(), f"Unknown diagnoses: {meta['diagnosis'][Y_TRUE.isna()].unique()}"
Y_TRUE = Y_TRUE.to_numpy().astype(int)

C_TRUE = np.zeros((len(meta), N_GROUPS), dtype=int)       # which value each lesion has, per criterion
for k, (group, values) in enumerate(CONCEPTS.items()):
    name_to_number = {name: i for i, names in enumerate(values.values()) for name in names}
    column = meta[group].map(name_to_number)
    assert column.notna().all(), f"Unknown values in {group}: {meta[group][column.isna()].unique()}"
    C_TRUE[:, k] = column.astype(int)

C_ONEHOT = np.zeros((len(meta), N_VALUES), dtype=np.float32)  # the same thing as 0s and 1s
for k, s in enumerate(SLICES):
    C_ONEHOT[np.arange(len(meta)), s.start + C_TRUE[:, k]] = 1

print("Cases where our checklist points differ from the dataset's own score:", int((C_ONEHOT @ POINTS != meta["seven_point_score"]).sum()))

# The split made by the dataset authors
SPLITS = {name: pd.read_csv(DATA_DIR / f"meta/{name}_indexes.csv")["indexes"].to_numpy()
          for name in ["train", "valid", "test"]}

### Where the data goes

We use the split made by the dataset authors, so our numbers can be compared with theirs.

| Part | Used for | Never used for |
|---|---|---|
| **Train** | Fitting all network weights and `f` | |
| **Validation** | Picking the best epoch, the strength of `f`, λ for the joint model, and the intervention order | Fitting weights |
| **Test** | Only the final numbers reported below | Any choice about the model |

The next cell checks that no lesion is in two parts, that every lesion is used, and that no image file appears in two parts.

In [ ]:
all_indexes = np.concatenate(list(SPLITS.values()))
assert len(all_indexes) == len(set(all_indexes)), "A case is in more than one split"
assert sorted(all_indexes) == list(range(len(meta))), "Some cases are not in any split"

paths = [DATA_DIR / "images" / name for name in meta[IMAGE_TYPE]]
hashes = [hashlib.md5(path.read_bytes()).hexdigest() for path in paths]
split_of = {i: name for name, idx in SPLITS.items() for i in idx}
hash_to_splits = {}
for i, file_hash in enumerate(hashes):
    hash_to_splits.setdefault(file_hash, set()).add(split_of[i])
print("Image files that appear in more than one split:", sum(len(s) > 1 for s in hash_to_splits.values()))
print("Columns that could identify a patient:", [c for c in meta.columns if "patient" in c.lower()] or "none")

counts = pd.DataFrame({name: pd.Series(Y_TRUE[idx]).value_counts().sort_index() for name, idx in SPLITS.items()})
counts.index = CLASS_NAMES
counts.loc["total"] = counts.sum()
display(counts)

The classes are very uneven, so we do two things:

- **Weighted training loss:** rare classes count more, so the model can't do well by ignoring them.
- **Balanced accuracy as the main score:** it is the average of the hit rate per class. Plain accuracy mostly rewards getting the most common class right.

**Limit to mention:** the metadata has no patient ID, so we cannot rule out one patient having lesions in two splits.

In [ ]:
def load_image(path):
    image = Image.open(path).convert("RGB")
    width, height = image.size
    image = image.crop((25, 25, width - 25, height - 25))  # many images have a black border
    image = TF.center_crop(TF.resize(image, 256, antialias=True), 256)
    return TF.pil_to_tensor(image)


IMAGES = torch.stack([load_image(path) for path in paths])
print("Images:", IMAGES.shape)

NORMALIZE = [v2.ToDtype(torch.float32, scale=True), v2.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])]
TRAIN_TRANSFORM = v2.Compose([v2.RandomResizedCrop(224, scale=(0.7, 1.0), antialias=True), v2.RandomHorizontalFlip(),
                              v2.RandomVerticalFlip(), v2.ColorJitter(0.1, 0.1, 0.1), *NORMALIZE])
TEST_TRANSFORM = v2.Compose([v2.CenterCrop(224), *NORMALIZE])


class Lesions(Dataset):
    def __init__(self, indexes, augment=False):
        self.indexes, self.transform = indexes, TRAIN_TRANSFORM if augment else TEST_TRANSFORM

    def __len__(self):
        return len(self.indexes)

    def __getitem__(self, i):
        j = self.indexes[i]
        return self.transform(IMAGES[j]), C_TRUE[j], Y_TRUE[j]

## 2. The model

A concept bottleneck model has two parts, and the second part **only** sees the output of the first:

```
image x  ->  g (ResNet-18)  ->  concepts c (19 numbers)  ->  f (one linear layer)  ->  diagnosis y
```

- **`g`** is a ResNet-18 pretrained on ImageNet. It gives a score per concept value, which we turn into probabilities per criterion. The paper also used ResNet-18 in one of its two tasks.
- **`f`** is a single linear layer, as in the paper's bird task. Its weights can be read directly: "this concept value pushes towards this diagnosis".

| Training | How `g` is trained | How `f` is trained |
|---|---|---|
| **Independent** | On true concepts | On **true** concepts. On test images it gets `g`'s predictions |
| **Sequential** | Same as independent | On **`g`'s predictions** on the training set, with `g` frozen |
| **Joint** | Together with `f` | Together with `g`, minimising `diagnosis loss + λ × concept loss` |

We also train a **standard** model with no concepts, to show what the bottleneck costs.

**Our choices that differ from the paper**

- `f` receives probabilities (0 to 1), not raw scores. All three models then get the same kind of input, and an intervention simply sets a value to 0 or 1.
- The concepts have several exclusive values, so each criterion uses one softmax and cross-entropy. The paper mostly uses yes/no concepts.
- The concept loss is averaged over the 7 criteria, so a given λ is not directly comparable to the paper's.

In [ ]:
def logits_to_probs(logits):
    """Turn concept scores into probabilities, separately for each criterion."""
    return torch.cat([logits[:, s].softmax(dim=1) for s in SLICES], dim=1)


class ConceptBottleneckModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.g = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
        self.g.fc = nn.Linear(self.g.fc.in_features, N_VALUES)
        self.f = nn.Linear(N_VALUES, N_CLASSES)

    def forward(self, x):
        concept_logits = self.g(x)
        diagnosis_logits = self.f(logits_to_probs(concept_logits))  # f only sees the concepts
        return concept_logits, diagnosis_logits


def make_standard_model():
    model = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
    model.fc = nn.Linear(model.fc.in_features, N_CLASSES)
    return model


def class_weights(labels, n):
    """Rare classes get a bigger weight. Computed on the training set only."""
    counts = np.bincount(labels, minlength=n).clip(min=1)
    return torch.tensor(len(labels) / (n * counts), dtype=torch.float32, device=DEVICE)


train_idx = SPLITS["train"]
DIAGNOSIS_WEIGHTS = class_weights(Y_TRUE[train_idx], N_CLASSES)
CONCEPT_WEIGHTS = [class_weights(C_TRUE[train_idx, k], s.stop - s.start) for k, s in enumerate(SLICES)]


def diagnosis_loss(diagnosis_logits, y):
    return F.cross_entropy(diagnosis_logits, y, weight=DIAGNOSIS_WEIGHTS)


def concept_loss(concept_logits, c):
    losses = [F.cross_entropy(concept_logits[:, s], c[:, k], weight=CONCEPT_WEIGHTS[k]) for k, s in enumerate(SLICES)]
    return torch.stack(losses).mean()

### Checks that this really is a concept bottleneck

These checks fail with an error if the model is built wrongly.

In [ ]:
set_seed(0)
_net = ConceptBottleneckModel().to(DEVICE)
_x, _c, _y = next(iter(DataLoader(Lesions(train_idx), batch_size=8)))
_x, _c, _y = _x.to(DEVICE), _c.to(DEVICE), _y.to(DEVICE)
_net.eval()
_logits, _out = _net(_x)

# 1. f takes exactly the concept values as input, and nothing else
assert _net.f.in_features == N_VALUES
assert torch.allclose(_out, _net.f(logits_to_probs(_net.g(_x))))

# 2. Each criterion is a proper set of probabilities that adds up to 1
_probs = logits_to_probs(_logits)
for s in SLICES:
    assert torch.allclose(_probs[:, s].sum(1), torch.ones(len(_x), device=DEVICE))

# 3. Changing the concepts changes the diagnosis (this is what makes interventions possible)
_true_c = torch.tensor(C_ONEHOT[train_idx[:8]], device=DEVICE)
assert not torch.allclose(_net.f(_true_c), _out)

# 4. Training g on concepts alone leaves f untouched (independent and sequential)
_net.zero_grad()
concept_loss(_net.g(_x), _c).backward()
assert _net.f.weight.grad is None and _net.g.fc.weight.grad is not None

# 5. In joint training, the diagnosis error does reach g
_net.zero_grad()
diagnosis_loss(_net(_x)[1], _y).backward()
assert _net.g.fc.weight.grad.abs().sum() > 0 and _net.f.weight.grad.abs().sum() > 0

del _net
print("All bottleneck checks passed")

## 3. Training

`train_network` is used for `g`, the joint model and the standard model. After each epoch it scores the model on validation data and keeps the best version.

`fit_f` trains `f` for the independent and sequential models. A linear layer on fixed inputs is just logistic regression, so we use scikit-learn for it. It tries a few strengths and keeps the best on validation data.

In [ ]:
def collect(function, indexes):
    """Run a function over images without training, and return the result as a numpy array."""
    outputs = []
    with torch.no_grad():
        for x, _, _ in DataLoader(Lesions(indexes), batch_size=64):
            outputs.append(function(x.to(DEVICE)).cpu())
    return torch.cat(outputs).numpy()


def train_network(name, seed, make_model, loss_function, validation_score):
    set_seed(seed)
    model = make_model().to(DEVICE)
    path = CACHE_DIR / f"{name}.pt"
    if path.exists():
        model.load_state_dict(torch.load(path, map_location=DEVICE, weights_only=True))
        return model.eval()

    def learning_rate(name):
        if name.startswith("f."):
            return LR_F
        return LR_NEW_LAYERS if name.endswith(("fc.weight", "fc.bias")) else LR_BACKBONE

    parameters = [{"params": [p], "lr": learning_rate(n)} for n, p in model.named_parameters()]
    optimizer = torch.optim.AdamW(parameters, weight_decay=WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, EPOCHS)
    loader = DataLoader(Lesions(SPLITS["train"], augment=True), batch_size=BATCH_SIZE, shuffle=True, drop_last=True)

    best_score, best_state, best_epoch = -1, None, None
    for epoch in range(EPOCHS):
        model.train()
        for x, c, y in loader:
            optimizer.zero_grad()
            loss_function(model, x.to(DEVICE), c.to(DEVICE), y.to(DEVICE)).backward()
            optimizer.step()
        scheduler.step()
        model.eval()
        score = validation_score(model)
        if score > best_score:
            best_score, best_epoch = score, epoch
            best_state = copy.deepcopy({k: v.cpu() for k, v in model.state_dict().items()})
    print(f"{name}: best validation score {best_score:.3f} at epoch {best_epoch + 1}/{EPOCHS}")
    torch.save(best_state, path)
    model.load_state_dict(best_state)
    return model.eval()


def concept_balanced_accuracy(probs, indexes):
    """Average balanced accuracy over the 7 criteria."""
    return np.mean([balanced_accuracy_score(C_TRUE[indexes, k], probs[:, s].argmax(1)) for k, s in enumerate(SLICES)])


def fit_f(c_train, y_train, c_valid, y_valid):
    best = {"score": -1}
    for strength in [0.01, 0.1, 1, 10, 100]:
        model = LogisticRegression(C=strength, class_weight="balanced", max_iter=5000).fit(c_train, y_train)
        score = balanced_accuracy_score(y_valid, model.predict(c_valid))
        if score > best["score"]:
            best = {"score": score, "W": model.coef_, "b": model.intercept_}
    return best


def predict_concepts(net):
    return {name: collect(lambda x: logits_to_probs(net.g(x)), idx) for name, idx in SPLITS.items()}

In [ ]:
valid_idx = SPLITS["valid"]
y_train, y_valid = Y_TRUE[train_idx], Y_TRUE[valid_idx]


def concept_only_loss(model, x, c, y):
    return concept_loss(model.g(x), c)


def score_concepts(model):
    return concept_balanced_accuracy(collect(lambda x: logits_to_probs(model.g(x)), valid_idx), valid_idx)


def score_diagnosis(model):
    return balanced_accuracy_score(y_valid, collect(lambda x: model(x)[1], valid_idx).argmax(1))


def train_joint(lam, seed):
    def loss(model, x, c, y):
        concept_logits, diagnosis_logits = model(x)
        return diagnosis_loss(diagnosis_logits, y) + lam * concept_loss(concept_logits, c)
    return train_network(f"joint_lambda{lam}_seed{seed}", seed, ConceptBottleneckModel, loss, score_diagnosis)


# Pick λ on validation data with the rule from the paper: the best diagnosis
# score among the models that keep a good concept score.
rows = []
for lam in LAMBDAS:
    net = train_joint(lam, SEEDS[0])
    probs = collect(lambda x: logits_to_probs(net.g(x)), valid_idx)
    rows.append({"lambda": lam, "valid diagnosis": score_diagnosis(net), "valid concepts": concept_balanced_accuracy(probs, valid_idx)})
lambda_table = pd.DataFrame(rows)
display(lambda_table.round(3))
good_concepts = lambda_table[lambda_table["valid concepts"] >= lambda_table["valid concepts"].max() - 0.05]
BEST_LAMBDA = float(good_concepts.loc[good_concepts["valid diagnosis"].idxmax(), "lambda"])
print("Chosen lambda:", BEST_LAMBDA)

In [ ]:
REGIMES = ["independent", "sequential", "joint"]
runs = {regime: [] for regime in REGIMES}   # for each seed: predicted concepts, and the weights of f
standard_probs = []                         # test predictions of the standard model

for seed in SEEDS:
    # Independent and sequential share the same g, as in the paper
    g_net = train_network(f"g_seed{seed}", seed, ConceptBottleneckModel, concept_only_loss, score_concepts)
    g_concepts = predict_concepts(g_net)

    # Independent: f learns from TRUE concepts, and is chosen on TRUE validation concepts, so it never sees g
    f_independent = fit_f(C_ONEHOT[train_idx], y_train, C_ONEHOT[valid_idx], y_valid)
    runs["independent"].append({"concepts": g_concepts, "W": f_independent["W"], "b": f_independent["b"]})

    # Sequential: f learns from g's PREDICTED concepts
    f_sequential = fit_f(g_concepts["train"], y_train, g_concepts["valid"], y_valid)
    runs["sequential"].append({"concepts": g_concepts, "W": f_sequential["W"], "b": f_sequential["b"]})

    # Joint: g and f are trained together
    joint_net = train_joint(BEST_LAMBDA, seed)
    runs["joint"].append({"concepts": predict_concepts(joint_net),
                          "W": joint_net.f.weight.detach().cpu().numpy(),
                          "b": joint_net.f.bias.detach().cpu().numpy()})

    # Standard: no concepts at all
    standard = train_network(f"standard_seed{seed}", seed, make_standard_model,
                             lambda model, x, c, y: diagnosis_loss(model(x), y),
                             lambda model: balanced_accuracy_score(y_valid, collect(model, valid_idx).argmax(1)))
    standard_probs.append(collect(lambda x: standard(x).softmax(1), SPLITS["test"]))

print("All models ready")

## 4. Standard performance assessment

Everything from here on uses the **test set**, and no more training happens. Numbers are the mean ± spread (standard deviation) over the seeds.

In [ ]:
test_idx = SPLITS["test"]
y_test = Y_TRUE[test_idx]


def diagnose(concepts, run):
    """Apply f by hand: weights x concepts + bias, then softmax."""
    scores = concepts @ run["W"].T + run["b"]
    scores = scores - scores.max(axis=1, keepdims=True)
    return np.exp(scores) / np.exp(scores).sum(axis=1, keepdims=True)


def macro_auc(y, probs):
    """How well the probabilities rank each class, averaged over the classes that appear."""
    aucs = [roc_auc_score(y == v, probs[:, v]) for v in range(probs.shape[1]) if 0 < (y == v).sum() < len(y)]
    return float(np.mean(aucs))


def diagnosis_metrics(y, probs):
    predicted = probs.argmax(1)
    return {"accuracy": accuracy_score(y, predicted),
            "balanced accuracy": balanced_accuracy_score(y, predicted),
            "macro F1": f1_score(y, predicted, average="macro", zero_division=0),
            "macro AUC": macro_auc(y, probs),
            "melanoma sensitivity": (predicted[y == MEL] == MEL).mean(),
            "melanoma specificity": (predicted[y != MEL] != MEL).mean()}


# Sanity check: f applied by hand must match the joint network's own output
_check = collect(lambda x: joint_net(x)[1].softmax(1), test_idx)
assert np.allclose(_check, diagnose(runs["joint"][-1]["concepts"]["test"], runs["joint"][-1]), atol=1e-3)

test_probs = {regime: [diagnose(run["concepts"]["test"], run) for run in runs[regime]] for regime in REGIMES}
test_probs["standard"] = standard_probs
ALL_MODELS = REGIMES + ["standard"]


def mean_spread(values):
    return f"{np.mean(values):.3f} ± {np.std(values):.3f}"


table = {model: pd.DataFrame([diagnosis_metrics(y_test, p) for p in test_probs[model]]).agg(mean_spread)
         for model in ALL_MODELS}
most_common = np.bincount(y_train).argmax()
table["always guess most common"] = pd.Series(diagnosis_metrics(y_test, np.eye(N_CLASSES)[np.full(len(y_test), most_common)])).round(3)
performance_table = pd.DataFrame(table).T
display(performance_table)

The seeds show how much training luck matters, but the test set is also small. To judge whether two models truly differ, we redraw the test set 1,000 times and look at the difference in balanced accuracy. If the 95% range includes 0, the difference could easily be chance.

In [ ]:
rng = np.random.default_rng(0)
resamples = [rng.integers(0, len(y_test), len(y_test)) for _ in range(1000)]
boot = {model: np.array([np.mean([balanced_accuracy_score(y_test[r], p[r].argmax(1)) for p in test_probs[model]])
                         for r in resamples]) for model in ALL_MODELS}

pairs = [("joint", "independent"), ("joint", "sequential"), ("sequential", "independent"),
         ("standard", "independent"), ("standard", "sequential"), ("standard", "joint")]
display(pd.DataFrame([{"comparison": f"{a} minus {b}", "difference": (boot[a] - boot[b]).mean(),
                       "95% low": np.percentile(boot[a] - boot[b], 2.5),
                       "95% high": np.percentile(boot[a] - boot[b], 97.5)} for a, b in pairs]).round(3))

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(13, 3.2), sharey=True)
for ax, model in zip(axes, ALL_MODELS):
    matrix = sum(confusion_matrix(y_test, p.argmax(1), labels=range(N_CLASSES)) for p in test_probs[model]).astype(float)
    matrix = matrix / matrix.sum(axis=1, keepdims=True)
    ax.imshow(matrix, cmap="Blues", vmin=0, vmax=1)
    ax.grid(False)
    for i in range(N_CLASSES):
        for j in range(N_CLASSES):
            ax.text(j, i, f"{matrix[i, j]:.2f}", ha="center", va="center", fontsize=8,
                    color="white" if matrix[i, j] > 0.6 else "#0b0b0b")
    ax.set_title(model)
    ax.set_xticks(range(N_CLASSES), CLASS_NAMES)
    ax.set_yticks(range(N_CLASSES), CLASS_NAMES)
    ax.set_xlabel("Predicted")
axes[0].set_ylabel("True diagnosis")
fig.suptitle("Share of each true diagnosis that gets each prediction (all seeds)", y=1.03)
plt.show()

recall = {model: [np.mean([(p.argmax(1)[y_test == v] == v).mean() for p in test_probs[model]]) for v in range(N_CLASSES)]
          for model in ALL_MODELS}
fig, ax = plt.subplots(figsize=(8, 3.2))
for i, model in enumerate(ALL_MODELS):
    ax.bar(np.arange(N_CLASSES) + (i - 1.5) * 0.2, recall[model], 0.18, label=model, color=COLORS[model])
ax.set_xticks(range(N_CLASSES), [f"{name} (n={(y_test == v).sum()})" for v, name in enumerate(CLASS_NAMES)])
ax.set_ylabel("Share found")
ax.set_ylim(0, 1)
ax.grid(axis="x", visible=False)
ax.set_title("How often each diagnosis is found on the test set")
ax.legend(frameon=False, ncol=4, loc="upper center", bbox_to_anchor=(0.5, -0.12))
plt.show()

**What do we see?** See `results.html`.

## 5. Exploring the explanations

The explanation of a CBM is its predicted concepts plus the weights of `f`. Here we ask whether the explanations help to explain the differences in performance.

### 5.1 How good are the predicted concepts?

Independent and sequential share `g`, so they have the same concepts. Only joint can differ. "Always guess" is the accuracy from always picking the most common value of that criterion.

In [ ]:
concept_rows = []
for k, (group, s) in enumerate(zip(GROUP_NAMES, SLICES)):
    truth = C_TRUE[test_idx, k]
    row = {"criterion": group, "always guess": (truth == np.bincount(C_TRUE[train_idx, k]).argmax()).mean()}
    for label, regime in [("g", "independent"), ("joint", "joint")]:
        predictions = [run["concepts"]["test"][:, s] for run in runs[regime]]
        row[f"{label} accuracy"] = np.mean([(p.argmax(1) == truth).mean() for p in predictions])
        row[f"{label} balanced"] = np.mean([balanced_accuracy_score(truth, p.argmax(1)) for p in predictions])
        row[f"{label} AUC"] = np.mean([macro_auc(truth, p) for p in predictions])
    concept_rows.append(row)
concept_table = pd.DataFrame(concept_rows).set_index("criterion")
concept_table.loc["average"] = concept_table.mean()
display(concept_table.round(3))

fig, ax = plt.subplots(figsize=(8, 3.4))
positions = np.arange(N_GROUPS)
ax.barh(positions - 0.2, concept_table["g balanced"][:N_GROUPS], 0.36, color=COLORS["independent"], label="independent and sequential (g)")
ax.barh(positions + 0.2, concept_table["joint balanced"][:N_GROUPS], 0.36, color=COLORS["joint"], label="joint")
ax.set_yticks(positions, GROUP_NAMES)
ax.invert_yaxis()
ax.set_xlim(0, 1)
ax.grid(axis="y", visible=False)
ax.set_xlabel("Balanced accuracy on the test set")
ax.set_title("How well each criterion is predicted")
ax.legend(frameon=False, ncol=2, loc="upper center", bbox_to_anchor=(0.5, -0.2))
plt.show()

### 5.2 Do correct concepts go with correct diagnoses?

If the explanations truly drive the prediction, correct diagnoses should come with more correct concepts.

We also make the concepts "hard" by setting the most likely value to 1 and the others to 0. If a model loses a lot from this, its diagnosis depends on fine detail in the probabilities, not on the concept values a person would read. Later work (Margeloiu et al., 2021; Mahinpei et al., 2021) warns that concepts can carry such extra information, which is called "leakage".

In [ ]:
def harden(probs):
    hard = np.zeros_like(probs)
    for s in SLICES:
        hard[np.arange(len(probs)), s.start + probs[:, s].argmax(1)] = 1
    return hard


rows = []
for regime in REGIMES:
    n_right, n_wrong, soft, hard = [], [], [], []
    for run in runs[regime]:
        concepts = run["concepts"]["test"]
        correct_concepts = sum((concepts[:, s].argmax(1) == C_TRUE[test_idx, k]).astype(int) for k, s in enumerate(SLICES))
        probs = diagnose(concepts, run)
        correct = probs.argmax(1) == y_test
        n_right.append(correct_concepts[correct].mean())
        n_wrong.append(correct_concepts[~correct].mean())
        soft.append(balanced_accuracy_score(y_test, probs.argmax(1)))
        hard.append(balanced_accuracy_score(y_test, diagnose(harden(concepts), run).argmax(1)))
    rows.append({"model": regime, "correct concepts (of 7) when diagnosis is right": np.mean(n_right),
                 "correct concepts (of 7) when diagnosis is wrong": np.mean(n_wrong),
                 "balanced accuracy, normal": np.mean(soft), "balanced accuracy, hard concepts": np.mean(hard),
                 "loss from hard concepts": np.mean(soft) - np.mean(hard)})
display(pd.DataFrame(rows).set_index("model").round(3).T)

**Do the predicted concepts carry extra diagnosis information?** A second leakage test, with three parts:

- Each model's own `f` given its predicted concepts, and then given the true concepts. A large gap suggests that `f` and `g` have adapted to each other, and that the concepts no longer mean what their names say.
- A new `f` fitted on the predicted concepts of the validation set and tested on the test set, which compares how much diagnosis information the predictions hold.

In [ ]:
rows = []
for regime in REGIMES:
    row = {}
    for split in ["train", "test"]:
        indexes = SPLITS[split]
        row[f"concepts balanced accuracy, {split}"] = np.mean([concept_balanced_accuracy(run["concepts"][split], indexes) for run in runs[regime]])
        row[f"diagnosis from predicted concepts, {split}"] = np.mean([balanced_accuracy_score(Y_TRUE[indexes], diagnose(run["concepts"][split], run).argmax(1)) for run in runs[regime]])
        row[f"diagnosis from true concepts, {split}"] = np.mean([balanced_accuracy_score(Y_TRUE[indexes], diagnose(C_ONEHOT[indexes], run).argmax(1)) for run in runs[regime]])
    new_f = [LogisticRegression(class_weight="balanced", max_iter=5000).fit(run["concepts"]["valid"], y_valid).predict(run["concepts"]["test"]) for run in runs[regime]]
    row["new f on predicted concepts, test"] = np.mean([balanced_accuracy_score(y_test, p) for p in new_f])
    rows.append(row)
display(pd.DataFrame(rows, index=REGIMES).round(3).T)

### 5.3 What has `f` learned?

Each square is a weight of `f`. Red pushes towards that diagnosis, and blue pushes away. We compare these with medical knowledge in two ways:

- the melanoma row against the points of the 7-point checklist, as a correlation
- the checklist itself (3 or more points means melanoma) as a simple baseline

In [ ]:
limit = max(np.abs(np.mean([run["W"] for run in runs[regime]], axis=0)).max() for regime in REGIMES)
fig, axes = plt.subplots(1, 3, figsize=(13, 5.2), sharey=True)
for ax, regime in zip(axes, REGIMES):
    mean_weights = np.mean([run["W"] for run in runs[regime]], axis=0).T
    image = ax.imshow(mean_weights, cmap="RdBu_r", vmin=-limit, vmax=limit, aspect="auto")
    ax.grid(False)
    ax.set_title(regime)
    ax.set_xticks(range(N_CLASSES), CLASS_NAMES)
    ax.set_yticks(range(N_VALUES), VALUE_NAMES, fontsize=8)
fig.colorbar(image, ax=axes, shrink=0.7, label="Weight of f (average over seeds)")
plt.show()

rows = [{"model": regime, "correlation of melanoma weights with checklist points":
         np.mean([np.corrcoef(run["W"][MEL], POINTS)[0, 1] for run in runs[regime]])} for regime in REGIMES]
display(pd.DataFrame(rows).set_index("model").round(3))


def checklist_rule(concepts):
    return harden(concepts) @ POINTS >= 3


def sensitivity_specificity(flagged):
    return {"melanoma sensitivity": flagged[y_test == MEL].mean(), "melanoma specificity": (~flagged[y_test != MEL]).mean()}


rule_rows = {"checklist on true concepts": sensitivity_specificity(checklist_rule(C_ONEHOT[test_idx]))}
for label, regime in [("checklist on g's concepts", "independent"), ("checklist on joint's concepts", "joint")]:
    rule_rows[label] = pd.DataFrame([sensitivity_specificity(checklist_rule(run["concepts"]["test"])) for run in runs[regime]]).mean().to_dict()
display(pd.DataFrame(rule_rows).T.round(3))

### 5.4 Are the explanations stable?

An explanation that changes with the random seed is hard to trust. For each pair of seeds we compare how often the predicted concept values match, and how often the same criterion contributes most to the predicted diagnosis.

In [ ]:
def top_criterion(concepts, run):
    """The criterion with the biggest push towards the predicted diagnosis, for each lesion."""
    predicted = diagnose(concepts, run).argmax(1)
    push = np.stack([(concepts[:, s] * run["W"][predicted][:, s]).sum(1) for s in SLICES], axis=1)
    return push.argmax(1)


rows = []
for regime in REGIMES:
    same_concepts, same_top, same_diagnosis = [], [], []
    for a in range(len(SEEDS)):
        for b in range(a + 1, len(SEEDS)):
            run_a, run_b = runs[regime][a], runs[regime][b]
            c_a, c_b = run_a["concepts"]["test"], run_b["concepts"]["test"]
            same_concepts.append((harden(c_a) * harden(c_b)).sum(1).mean() / N_GROUPS)
            same_top.append((top_criterion(c_a, run_a) == top_criterion(c_b, run_b)).mean())
            same_diagnosis.append((diagnose(c_a, run_a).argmax(1) == diagnose(c_b, run_b).argmax(1)).mean())
    rows.append({"model": regime, "same concept values": np.mean(same_concepts),
                 "same top criterion": np.mean(same_top), "same diagnosis": np.mean(same_diagnosis)})
display(pd.DataFrame(rows).set_index("model").round(3))

### 5.5 Examples where the models disagree

These are test lesions where the three models give different diagnoses, using the first seed. Melanoma cases are shown first. Compare the concepts: which model is wrong, and can its concepts explain why?

In [ ]:
diagnoses = {regime: test_probs[regime][0].argmax(1) for regime in REGIMES}
disagree = np.where((diagnoses["independent"] != diagnoses["sequential"]) | (diagnoses["independent"] != diagnoses["joint"]))[0]
print(f"The models disagree on {len(disagree)} of {len(y_test)} test lesions (first seed)")
examples = sorted(disagree, key=lambda i: y_test[i] != MEL)[:4]

for i in examples:
    case = test_idx[i]
    plt.figure(figsize=(2.5, 2.5))
    plt.imshow(IMAGES[case].permute(1, 2, 0).numpy())
    plt.axis("off")
    plt.title(f"True diagnosis: {CLASS_NAMES[Y_TRUE[case]]}", fontsize=10)
    plt.show()
    rows = {}
    for k, (group, s) in enumerate(zip(GROUP_NAMES, SLICES)):
        values = list(CONCEPTS[group])
        row = {"true": values[C_TRUE[case, k]]}
        for label, regime in [("g", "independent"), ("joint", "joint")]:
            p = runs[regime][0]["concepts"]["test"][i, s]
            row[label] = f"{values[p.argmax()]} ({p.max():.2f})"
        rows[group] = row
    rows["diagnosis"] = {"true": CLASS_NAMES[Y_TRUE[case]],
                         "g": " / ".join(f"{r}: {CLASS_NAMES[diagnoses[r][i]]}" for r in ["independent", "sequential"]),
                         "joint": CLASS_NAMES[diagnoses["joint"][i]]}
    display(pd.DataFrame(rows).T)

**Do the explanations support the performance differences?** See `results.html`.

## 6. Interventions

Here we act as an expert who corrects the model. For a test lesion, we replace the predicted values of some criteria with the true ones (true value = 1, other values = 0) and let `f` make a new diagnosis. We increase the number of corrected criteria from 0 to 7.

We use both orders from the paper:

- **Random order**, averaged over many random orders (the paper's bird task).
- **Best first**, a fixed order worked out on **validation** data, starting with the criterion that helps most on its own (the paper's knee task).

As in the paper, we correct whole criteria at a time. The paper's bird task sets raw scores to the 5th or 95th percentile for sequential and joint models. We can use 0 and 1 directly because our `f` takes probabilities.

In [ ]:
def intervene(concepts, indexes, criteria):
    fixed = concepts.copy()
    for k in criteria:
        fixed[:, SLICES[k]] = C_ONEHOT[indexes][:, SLICES[k]]
    return fixed


def score_after_intervention(run, split, criteria, metric):
    indexes = SPLITS[split]
    return metric(Y_TRUE[indexes], diagnose(intervene(run["concepts"][split], indexes, criteria), run).argmax(1))


def melanoma_sensitivity(y, predicted):
    return (predicted[y == MEL] == MEL).mean()


def best_first_order(run):
    gains = [score_after_intervention(run, "valid", [k], balanced_accuracy_score) for k in range(N_GROUPS)]
    return list(np.argsort(gains)[::-1])


def intervention_curves(metric):
    random_curves, best_curves = {}, {}
    for regime in REGIMES:
        random_seed, best_seed = [], []
        for run in runs[regime]:
            rng = np.random.default_rng(0)  # the same random orders for each model
            orders = [rng.permutation(N_GROUPS) for _ in range(N_RANDOM_ORDERS)]
            random_seed.append([np.mean([score_after_intervention(run, "test", order[:n], metric) for order in orders])
                                for n in range(N_GROUPS + 1)])
            order = best_first_order(run)
            best_seed.append([score_after_intervention(run, "test", order[:n], metric) for n in range(N_GROUPS + 1)])
        random_curves[regime] = np.mean(random_seed, axis=0)
        best_curves[regime] = np.mean(best_seed, axis=0)
    return random_curves, best_curves


def plot_curves(metric, label):
    random_curves, best_curves = intervention_curves(metric)
    standard_score = np.mean([metric(y_test, p.argmax(1)) for p in standard_probs])
    fig, axes = plt.subplots(1, 2, figsize=(11, 3.8), sharey=True)
    for ax, curves, title in [(axes[0], random_curves, "Random order"), (axes[1], best_curves, "Best first (order from validation)")]:
        for regime in REGIMES:
            ax.plot(range(N_GROUPS + 1), curves[regime], marker="o", markersize=5, linewidth=2,
                    color=COLORS[regime], label=regime)
        ax.axhline(standard_score, color=COLORS["standard"], linestyle="--", linewidth=1.5, label="standard (no concepts)")
        ax.set_title(title)
        ax.set_xlabel("Number of criteria corrected")
        ax.grid(axis="x", visible=False)
    axes[0].set_ylabel(label)
    handles, labels = axes[0].get_legend_handles_labels()
    fig.legend(handles, labels, frameon=False, ncol=4, loc="lower center", bbox_to_anchor=(0.5, -0.12))
    plt.show()
    display(pd.DataFrame({f"{regime}, random": random_curves[regime] for regime in REGIMES}
                         | {f"{regime}, best first": best_curves[regime] for regime in REGIMES},
                         index=pd.Index(range(N_GROUPS + 1), name="criteria corrected")).round(3))


plot_curves(balanced_accuracy_score, "Balanced accuracy on the test set")
plot_curves(melanoma_sensitivity, "Melanoma sensitivity on the test set")

print("Best-first order (first seed):")
for regime in REGIMES:
    print(f"  {regime}:", [GROUP_NAMES[k] for k in best_first_order(runs[regime][0])])

**What happens when we intervene?** See `results.html`.

## 7. Use cases and success criteria

**This is a draft to start your group discussion.** The assignment asks you to agree on these together, so change them freely. The target numbers are suggestions.

| # | Use case | Measurable success criterion | How to measure it |
|---|---|---|---|
| 1 | **Help doctors decide when to trust the model.** A doctor sees the predicted criteria and can spot a flawed prediction. | With explanations, doctors follow the model more often when it is right and less often when it is wrong. Melanoma sensitivity also rises by at least 5 points, and specificity does not fall. | A study where doctors review the same type of cases with and without explanations. |
| 2 | **Let doctors correct the model.** A doctor fixes a criterion the model got wrong, and the diagnosis updates. | Correcting at most 2 criteria raises balanced accuracy by at least 5 points, and doctors need less than 1 extra minute per case. | Offline: the curves in section 6. In a study: the same measure using real doctors' corrections, which unlike ours may contain mistakes, along with the time taken. |
| 3 | **Find faults before the model is used.** A reviewer uses the explanations to find out if the model relies on something wrong, or works worse for some groups. | Reviewers find at least 80% of faults that were planted on purpose (such as a ruler or ink mark linked to melanoma), and more than reviewers without explanations. | A test where some models have planted faults, and reviewers don't know which ones. |

**Our final use cases:** *to be agreed in the group discussion.*